# PRUEBAS

In [0]:
%sql
--USE CATALOG sbsrisk_dev;
--USE SCHEMA bronze;

--CREATE TABLE IF NOT EXISTS bronze_sbs_indice_2024 (
--  _c0 STRING, _c1 STRING, _c2 STRING, _c3 STRING, _c4 STRING, _c5 STRING, _c6 STRING,
--  source_file STRING,
--  periodo_mes STRING
--)
--USING DELTA
--LOCATION 'abfss://bronze@adlsbsriskdev.dfs.core.windows.net/sbsrisk_dev/bronze/bronze_sbs_indice_2024';

In [0]:
"""
import re
from pyspark.sql.functions import lit

# Ajusta esta base al path REAL donde dejaste los excels
base_2024 = "abfss://raw@adlsbsriskdev.dfs.core.windows.net/sbs/2024/"

# Lista archivos (si tu carpeta es distinta, acá lo verás)
files = [f.path for f in dbutils.fs.ls(base_2024) if f.path.lower().endswith(".xls") or f.path.lower().endswith(".xlsx")]
print("Archivos encontrados:", len(files))
for f in files[:5]:
    print(f)

all_dfs = []

for file_path in files:
    file_name = file_path.split("/")[-1]

    # extrae mes: en2024, fe2024, etc.
    m = re.search(r"-([a-z]{2})\d{4}", file_name.lower())
    periodo = m.group(1) if m else "unknown"

    df = (
        spark.read.format("com.crealytics.spark.excel")
        .option("header", "false")
        .option("inferSchema", "false")
        .option("treatEmptyValuesAsNulls", "true")
        .option("sheetName", "INDICE")
        .load(file_path)
    )

    df = (
        df.withColumn("source_file", lit(file_name))
          .withColumn("periodo_mes", lit(periodo))
    )

    all_dfs.append(df)

# unir todo
df_final = all_dfs[0]
for d in all_dfs[1:]:
    df_final = df_final.unionByName(d)

display(df_final.limit(20))
"""

In [0]:
"""
(
  df_final.write.format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("sbsrisk_dev.bronze.bronze_sbs_indice_2024")
)
"""

In [0]:
"""
%sql
SELECT COUNT(*) AS total_rows
FROM sbsrisk_dev.bronze.bronze_sbs_indice_2024;
"""

In [0]:
# probando
# dbutils.fs.ls("abfss://raw@adlsbsriskdev.dfs.core.windows.net/")

In [0]:
#probando
# dbutils.fs.ls("abfss://raw@adlsbsriskdev.dfs.core.windows.net/sbs/")

In [0]:
# probando
# dbutils.fs.ls("abfss://raw@adlsbsriskdev.dfs.core.windows.net/sbs/2024/")

# INICIANDO

In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("storageName", "adlsbsriskdev")
dbutils.widgets.text("catalogName", "sbsrisk_dev")
dbutils.widgets.text("year", "2024")   # cambia a 2025 si vas a procesar ese año

storage = dbutils.widgets.get("storageName")
catalog = dbutils.widgets.get("catalogName")
year = dbutils.widgets.get("year")

spark.sql(f"USE CATALOG {catalog}")
spark.sql("USE SCHEMA bronze")

In [0]:
raw_base = f"abfss://raw@{storage}.dfs.core.windows.net/sbs/{year}/"
display(dbutils.fs.ls(raw_base))

In [0]:
files = [f.path for f in dbutils.fs.ls(raw_base) 
         if f.path.lower().endswith(".xls") or f.path.lower().endswith(".xlsx")]

print("Total excels encontrados:", len(files))
for f in files[:5]:
    print(f)

if len(files) == 0:
    raise Exception(f"No se encontraron archivos .XLS/.XLSX en: {raw_base}")

In [0]:
import re
from pyspark.sql.functions import lit

def read_excel_any_sheet(path: str):
    sheet_candidates = ["INDICE", "ÍNDICE", "INDICE ", "ÍNDICE ", "INDICE GENERAL", "ÍNDICE GENERAL"]
    last_err = None
    
    for sh in sheet_candidates:
        try:
            df = (spark.read.format("com.crealytics.spark.excel")
                  .option("header", "false")
                  .option("inferSchema", "false")
                  .option("treatEmptyValuesAsNulls", "true")
                  .option("sheetName", sh)
                  .load(path))
            return df
        except Exception as e:
            last_err = e
    
    # Fallback: intenta leer sin sheetName (primera hoja)
    try:
        df = (spark.read.format("com.crealytics.spark.excel")
              .option("header", "false")
              .option("inferSchema", "false")
              .option("treatEmptyValuesAsNulls", "true")
              .load(path))
        return df
    except Exception as e:
        raise Exception(f"No pude leer el Excel {path}. Último error: {last_err} / fallback error: {e}")

In [0]:
all_dfs = []

for file_path in files:
    file_name = file_path.split("/")[-1]

    # Extrae mes: en2024, fe2024, etc (en tu nombre es -en2024, -fe2024...)
    m = re.search(r"-([a-z]{2})\d{4}", file_name.lower())
    periodo = m.group(1) if m else "unknown"

    df = read_excel_any_sheet(file_path)

    df = (df.withColumn("source_file", lit(file_name))
            .withColumn("periodo_mes", lit(periodo)))

    all_dfs.append(df)

if len(all_dfs) == 0:
    raise Exception("La lista all_dfs quedó vacía (no se pudo leer ningún excel).")

df_final = all_dfs[0]
for d in all_dfs[1:]:
    df_final = df_final.unionByName(d, allowMissingColumns=True)

display(df_final.limit(20))
print("Total filas (aprox):", df_final.count())

In [0]:
bronze_path = f"abfss://bronze@{storage}.dfs.core.windows.net/{catalog}/bronze/bronze_sbs_indice_{year}"

(df_final.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(bronze_path))

print("OK escrito en:", bronze_path)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.bronze.bronze_sbs_indice_{year}
USING DELTA
LOCATION '{bronze_path}'
""")

spark.sql(f"SELECT COUNT(*) AS total_rows FROM {catalog}.bronze.bronze_sbs_indice_{year}").show()

In [0]:
display(dbutils.fs.ls(bronze_path))